# 08 · Event-study: hari kejutan berita
Tidak ada model. Hari "kejutan" = z-score kausal (statistik dari hari sebelumnya, minimal 10 hari) melewati ambang yang ditetapkan sebelumnya.
Bandingkan hasil di hari kejutan vs lainnya. p-value dari **pergeseran sirkular** deret kejutan (menjaga autokorelasi), koreksi Holm untuk 9 uji.
Pembacaan: `|ret_t|` = hari yang sama (kontemporer), `ret_t+1` dan `|ret_t+1|` = besok (yang relevan untuk prediksi).

In [1]:
import numpy as np, pandas as pd
from utils import *

df, _ = load(with_pup=False)

def z(x):                                              # kausal: rata-rata dan std hanya dari hari sebelum t
    p = x.shift(1)
    return (x - p.expanding(10).mean()) / p.expanding(10).std()

SHOCKS = {'news_count z>1': z(df.news_count) > 1, 'gpr_max z>1': z(df.gpr_max) > 1, 'sent_min z<-1': z(df.sent_min) < -1}
OUT = {'|ret_t|': df.target0.abs(), 'ret_t+1': df.target, '|ret_t+1|': df.target.abs()}
valid = z(df.news_count).notna().values
print('hari dipakai:', valid.sum(), '| hari kejutan:', {k: int(v[valid].sum()) for k, v in SHOCKS.items()})

hari dipakai: 131 | hari kejutan: {'news_count z>1': 15, 'gpr_max z>1': 20, 'sent_min z<-1': 0}


In [2]:
def stat(s, o): return o[s].mean() - o[~s].mean()

rows = []
for sn, s in SHOCKS.items():
    s, n = s.values[valid], valid.sum()
    for on, o in OUT.items():
        o = o.values[valid]
        obs = stat(s, o)
        perm = np.array([stat(np.roll(s, k), o) for k in range(5, n - 5)])
        rows.append(dict(kejutan=sn, hasil=on, selisih=obs, p=(np.abs(perm) >= abs(obs)).mean()))
R = pd.DataFrame(rows)
order = R.p.argsort().values; m = len(R)                # Holm
adj = np.empty(m); run_max = 0
for rank, i in enumerate(order):
    run_max = max(run_max, (m - rank) * R.p.iloc[i]); adj[i] = min(1, run_max)
R['p_holm'] = adj
print(R.round(6).to_string(index=False))
print('\nKejutan harus cukup sering (>=15 hari) dan p_holm < 0.05 baru layak disebut. Sisanya: tidak ada bukti.')

       kejutan     hasil   selisih        p   p_holm
news_count z>1   |ret_t| -0.000875 0.140496 0.561983
news_count z>1   ret_t+1  0.000309 0.809917 1.000000
news_count z>1 |ret_t+1|  0.001142 0.066116 0.330579
   gpr_max z>1   |ret_t| -0.000149 0.776860 1.000000
   gpr_max z>1   ret_t+1  0.000778 0.355372 1.000000
   gpr_max z>1 |ret_t+1|  0.001242 0.008264 0.049587
 sent_min z<-1   |ret_t|       NaN 0.000000 0.000000
 sent_min z<-1   ret_t+1       NaN 0.000000 0.000000
 sent_min z<-1 |ret_t+1|       NaN 0.000000 0.000000

Kejutan harus cukup sering (>=15 hari) dan p_holm < 0.05 baru layak disebut. Sisanya: tidak ada bukti.


C:\Users\ASUS\AppData\Local\Temp\ipykernel_16436\2111288809.py:1: RuntimeWarning: Mean of empty slice
  def stat(s, o): return o[s].mean() - o[~s].mean()
C:\Users\ASUS\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
